# 01 · Retrieval: BM25 vs Dense vs Hybrid vs Reranker

**Trước khi chạy** (khung Settings bên phải):
1. **Accelerator:** GPU T4 x2 (hoặc T4 x1)
2. **Internet:** On (tài khoản cần xác minh số điện thoại)

Sau đó bấm **Run All**. Mất khoảng 40–60 phút, chủ yếu để encode khoảng 106K chunk bằng bge-m3.

In [1]:
REPO_URL = "https://github.com/sinhtruc24/Vietnamese-Legal-RAG.git"
WORK = "/kaggle/working/Vietnamese-Legal-RAG"

import os, socket
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit("Không có Internet: Settings (bên phải) -> Internet = On, "
                     "rồi Run -> Restart & clear cell outputs và chạy lại.")

if not os.path.exists(WORK):
    !git clone -q {REPO_URL} {WORK}
else:
    !git -C {WORK} pull -q
assert os.path.exists(WORK), "Clone thất bại: kiểm tra REPO_URL và repo có để Public không"
%cd {WORK}
!git log --oneline -1

/kaggle/working/Vietnamese-Legal-RAG
a04d2d7 (HEAD -> main, origin/main, origin/HEAD) Rate limiting and daily-quota handling for free-tier teacher APIs


In [2]:
# Kaggle đã có sẵn torch + CUDA; chỉ cài thêm phần còn thiếu
!pip install -q -e ".[ml]" openai huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.1 MB/s eta 0:00:00
  Building editable for legalrag (pyproject.toml) ... done
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 1. Tải dữ liệu Zalo AI 2021 Legal

In [3]:
!python scripts/download_data.py

corpus.jsonl: 100%|██████████████████████████| 115M/115M [00:02<00:00, 50.7MB/s]
saved data/zalo_legal/corpus.jsonl
queries.jsonl: 100%|██████████████████████████| 552k/552k [00:00<00:00, 901kB/s]
saved data/zalo_legal/queries.jsonl
qrels/train.jsonl: 100%|██████████████████████| 231k/231k [00:00<00:00, 377kB/s]
saved data/zalo_legal/qrels/train.jsonl
qrels/test.jsonl: 100%|█████████████████████| 73.1k/73.1k [00:00<00:00, 177kB/s]
saved data/zalo_legal/qrels/test.jsonl

articles: 61,425  queries: 3,196
 test: 788 queries with gold articles
train: 2,432 queries with gold articles


## 2. Build index: BM25 + bigram và dense bge-m3
Nếu bị hết bộ nhớ GPU (OOM), giảm `--batch-size` xuống 32 hoặc 16.

In [4]:
!python scripts/build_index.py --bigrams --index-dir indexes/full --batch-size 64

61,425 articles -> 105,917 chunks
BM25 built in 38.3 s
config_sentence_transformers.json: 100%|████████| 123/123 [00:00<00:00, 621kB/s]
README.md: 15.8kB [00:00, 34.8MB/s]
sentence_bert_config.json: 100%|██████████████| 54.0/54.0 [00:00<00:00, 266kB/s]
config.json: 100%|█████████████████████████████| 687/687 [00:00<00:00, 4.66MB/s]
pytorch_model.bin: 100%|████████████████████| 2.27G/2.27G [00:06<00:00, 343MB/s]
Loading weights: 100%|█| 391/391 [00:00<00:00, 2486.89it/s, Materializing param=
tokenizer_config.json: 100%|███████████████████| 444/444 [00:00<00:00, 2.05MB/s]
model.safetensors:   0%|                            | 0.00/2.27G [00:01<?, ?B/s]
model.safetensors:   6%|█▏                   | 134M/2.27G [00:01<00:03, 669MB/s]
model.safetensors:   9%|█▊                   | 201M/2.27G [00:01<00:04, 473MB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 11.8MB/s]
model.safetensors:  12%|██▍                  | 268M/2.27G [00:01<00:04, 410MB/s]
tokenizer.json:   

## 3. Đánh giá retrieval trên 788 câu hỏi test

In [5]:
!python scripts/eval_retrieval.py --index-dir indexes/full \
    --methods bm25 dense hybrid hybrid+rerank dense+rerank \
    --output results/retrieval_test.json

test: 788 queries, 61,425 indexed articles
bm25: 100%|█████████████████████████████████████| 13/13 [00:01<00:00,  9.93it/s]
Loading weights: 100%|█| 391/391 [00:00<00:00, 3267.67it/s, Materializing param=
config.json: 100%|█████████████████████████████| 795/795 [00:00<00:00, 2.62MB/s]
model.safetensors: 100%|████████████████████| 2.27G/2.27G [00:06<00:00, 333MB/s]
Loading weights: 100%|█| 393/393 [00:00<00:00, 1414.82it/s, Materializing param=
tokenizer_config.json: 1.17kB [00:00, 5.17MB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 12.3MB/s]
tokenizer.json: 100%|██████████████████████| 17.1M/17.1M [00:00<00:00, 41.4MB/s]
dense+rerank: 100%|█████████████████████████████| 13/13 [05:24<00:00, 24.94s/it]

| method | recall@1 | recall@5 | recall@10 | mrr@10 | ndcg@10 | ms_per_query |
|---|---|---|---|---|---|---|
| bm25 | 0.6256 | 0.8522 | 0.8902 | 0.7258 | 0.7661 | 1.7 |
| dense | 0.6263 | 0.8725 | 0.9251 | 0.7325 | 0.7793 | 17.3 |
| hybrid | 0.6865 | 0.8953 | 

In [6]:
import json, pandas as pd
res = json.load(open("results/retrieval_test.json"))["results"]
pd.DataFrame(res).T.round(4)

,recall@1,recall@5,recall@10,mrr@10,ndcg@10,ms_per_query
bm25,0.6256,0.8522,0.8902,0.7258,0.7661,1.6699
dense,0.6263,0.8725,0.9251,0.7325,0.7793,17.2851
hybrid,0.6865,0.8953,0.9378,0.7794,0.8182,18.8004
hybrid+rerank,0.6967,0.9321,0.9632,0.7998,0.8395,410.3112
dense+rerank,0.6980,0.9194,0.9486,0.7945,0.8317,411.4802


## 4. Ablation: số ứng viên đưa vào reranker
Reranker chính xác hơn nhưng chậm hơn. Mục này tìm điểm cân bằng giữa chất lượng và độ trễ.

In [7]:
for n in [10, 20, 50]:
    !python scripts/eval_retrieval.py --index-dir indexes/full --methods hybrid+rerank \
        --rerank-candidates {n} --output results/retrieval_rerank{n}.json

test: 788 queries, 61,425 indexed articles
Loading weights: 100%|█| 391/391 [00:00<00:00, 2880.06it/s, Materializing param=
Loading weights: 100%|█| 393/393 [00:00<00:00, 1481.98it/s, Materializing param=
hybrid+rerank: 100%|████████████████████████████| 13/13 [02:06<00:00,  9.76s/it]

| method | recall@1 | recall@5 | recall@10 | mrr@10 | ndcg@10 | ms_per_query |
|---|---|---|---|---|---|---|
| hybrid+rerank | 0.6942 | 0.9175 | 0.9378 | 0.7922 | 0.8278 | 161.1 |

saved results/retrieval_rerank10.json
test: 788 queries, 61,425 indexed articles
Loading weights: 100%|█| 391/391 [00:00<00:00, 3718.51it/s, Materializing param=
Loading weights: 100%|█| 393/393 [00:00<00:00, 1682.43it/s, Materializing param=
hybrid+rerank: 100%|████████████████████████████| 13/13 [03:46<00:00, 17.40s/it]

| method | recall@1 | recall@5 | recall@10 | mrr@10 | ndcg@10 | ms_per_query |
|---|---|---|---|---|---|---|
| hybrid+rerank | 0.6954 | 0.9283 | 0.9594 | 0.7981 | 0.8372 | 287.1 |

saved results/retrieval_re

## 5. Lưu kết quả
- Bấm **Save Version → Save & Run All (Commit)** để lưu thư mục `/kaggle/working` (gồm index và kết quả).
- Mở tab **Output** của version đã lưu và tải `results/*.json` về máy để điền vào README.
- Muốn dùng lại index ở notebook khác: trong notebook đó chọn **Add Input → Notebook Output**, rồi chọn notebook này.